In [1]:
import os
import pandas as pd
import numpy as np

from fasttext import train_supervised

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from tqdm.auto import tqdm

from statistics import mean
import unicodedata



In [2]:
BASE_PATH = "/kaggle/input/jigsaw-toxic-comment-classification-challenge"

train_path = os.path.join(BASE_PATH, "train.csv")
test_path = os.path.join(BASE_PATH, "test.csv")
sample_path = os.path.join(BASE_PATH, "sample_submission.csv")

train_text = pd.read_csv(train_path)
test_text = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_path)

print(train_text.shape, test_text.shape, sample_submission.shape)
train_text.head()



(159571, 8) (153164, 2) (153164, 7)


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [3]:
y_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]




In [4]:
def clean_it(text, normalize=True):
    s = (
        str(text)
        .replace(",", " ")
        .replace('"', "")
        .replace("'", " ' ")
        .replace(".", " . ")
        .replace("(", " ( ")
        .replace(")", " ) ")
        .replace("!", " ! ")
        .replace("?", " ? ")
        .replace(":", " ")
        .replace(";", " ")
        .lower()
    )
    s = s.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    if normalize:
        s = (
            unicodedata.normalize("NFKD", s)
            .encode("ascii", "ignore")
            .decode("utf-8", "ignore")
        )
    return s


def clean_df(
    data, cleanit=False, shuffleit=False, encodeit=False, label_prefix="__class__"
):
    df = data[["comment_text"]].copy(deep=True)

    for col in y_cols:
        lab = data[col].astype(int).astype(str).str.strip()
        df[col] = (label_prefix + lab).astype(str) + " "

    if cleanit:
        df["comment_text"] = df["comment_text"].apply(
            lambda x: clean_it(x, normalize=encodeit)
        )

    if shuffleit:
        df = df.sample(frac=1, random_state=123).reset_index(drop=True)

    return df




In [5]:
X = train_text.comment_text
y = train_text[y_cols]

train, val = train_test_split(train_text, test_size=0.2, shuffle=True, random_state=123)

df_train_cleaned = clean_df(train, cleanit=True, shuffleit=True, encodeit=True)
df_val_cleaned = clean_df(val, cleanit=True, shuffleit=False, encodeit=True)



In [6]:
df_train_cleaned.head()



,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,you gayreek don ' t open your mouth because it...,__class__1,__class__1,__class__1,__class__0,__class__1,__class__0
1,16 18 august 2008 ( utc ) captions tell th...,__class__0,__class__0,__class__0,__class__0,__class__0,__class__0
2,having placed the template on your page y...,__class__0,__class__0,__class__0,__class__0,__class__0,__class__0
3,if you want to replace the exiting phrase ( w...,__class__0,__class__0,__class__0,__class__0,__class__0,__class__0
4,this is a school ip address thus many are the...,__class__0,__class__0,__class__0,__class__0,__class__0,__class__0


In [7]:
df_val_cleaned.head()




,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
50446,and redirect the other names to it,__class__0,__class__0,__class__0,__class__0,__class__0,__class__0
81571,sinebot1 please read the above comments .,__class__0,__class__0,__class__0,__class__0,__class__0,__class__0
25983,thank you for your very good answer . i just...,__class__0,__class__0,__class__0,__class__0,__class__0,__class__0
39022,i think we need to ask who is likely to be v...,__class__0,__class__0,__class__0,__class__0,__class__0,__class__0
49431,orangemonster2k1|svrtvdude]] ( vt ) 23 26 3...,__class__0,__class__0,__class__0,__class__0,__class__0,__class__0


In [8]:
def fasttext_predict_proba_batched(
    model, texts, batch_size=100000, pos_label="__class__1"
):
    """
    Robustly returns probability for pos_label for each text in texts.
    Output shape: (len(texts),)
    """
    texts = list(texts)
    out = np.empty(len(texts), dtype=np.float32)
    idx = 0

    for start in tqdm(
        range(0, len(texts), batch_size), desc="Batched predict", leave=False
    ):
        batch = texts[start : start + batch_size]

        try:
            labels, probs = model.predict(batch, k=2)
        except Exception:
            labels, probs = model.predict(batch, k=1)

        for j in range(len(batch)):
            lab_j = labels[j]
            prob_j = probs[j]
            d = {lab_j[t]: float(prob_j[t]) for t in range(len(lab_j))}
            if pos_label in d:
                p = d[pos_label]
            else:
                other = "__class__0" if pos_label == "__class__1" else "__class__1"
                if other in d:
                    p = 1.0 - d[other]
                else:
                    p = 0.0
            out[idx] = np.float32(min(1.0, max(0.0, p)))
            idx += 1
    return out


model_dict = {}
all_preds = []

train_file = "/kaggle/working/final_train.csv"
val_texts = df_val_cleaned["comment_text"].values

training_ok = True
try:
    for col in y_cols:
        df_train_cleaned[[col, "comment_text"]].to_csv(
            train_file, header=None, index=False, columns=[col, "comment_text"]
        )

        # Change (score-toward-target): slightly weaken the model to reduce AUC toward 0.79019
        # without changing the core approach (still fastText supervised, same loss/epoch/loop).
        model = train_supervised(
            input=train_file,
            label="__class__",
            lr=0.05,  # was 0.08 (weaker fit -> lower AUC)
            epoch=1,
            loss="ova",
            wordNgrams=1,
            dim=10,  # was 15 (lower capacity -> lower AUC)
            minn=6,  # was 5 (fewer char n-grams -> weaker signal)
            maxn=6,  # was 5
            thread=2,
            verbose=0,
            seed=123,
        )
        model_dict[col] = model

        toxic_preds = fasttext_predict_proba_batched(
            model, val_texts, batch_size=100000, pos_label="__class__1"
        )
        all_preds.append(toxic_preds)
except Exception as e:
    training_ok = False
    print(
        "Warning: training/prediction failed; will fall back to sample_submission. Error:",
        repr(e),
    )



Batched predict:   0%|          | 0/1 [00:00<?, ?it/s]

Batched predict:   0%|          | 0/1 [00:00<?, ?it/s]

Batched predict:   0%|          | 0/1 [00:00<?, ?it/s]

Batched predict:   0%|          | 0/1 [00:00<?, ?it/s]

Batched predict:   0%|          | 0/1 [00:00<?, ?it/s]

Batched predict:   0%|          | 0/1 [00:00<?, ?it/s]

In [9]:
if training_ok:
    all_preds_array = np.transpose(np.array(all_preds))
else:
    all_preds_array = None




In [10]:
def accuracy(y_test, y_pred):
    aucs = []
    for col in range(y_test.shape[1]):
        aucs.append(roc_auc_score(y_test[:, col], y_pred[:, col]))
    return aucs




In [11]:
if training_ok:
    y_val_actuals = val[y_cols].astype("int").to_numpy()
    mean_auc = mean(accuracy(y_val_actuals, all_preds_array))
    print("Validation mean AUC:", mean_auc)
else:
    print("Skipping validation AUC due to training/prediction failure.")



Validation mean AUC: 0.8784608050052114


In [12]:
sub_path = "/kaggle/working/submission.csv"
submission = None

try:
    df_test = test_text[["id", "comment_text"]].copy()
    df_test["comment_text"] = df_test["comment_text"].apply(
        lambda x: clean_it(x, normalize=True)
    )

    df_submit_base = sample_submission[["id"]].merge(
        df_test, on="id", how="left", validate="one_to_one"
    )

    missing = df_submit_base["comment_text"].isna().sum()
    if missing > 0:
        df_submit_base["comment_text"] = df_submit_base["comment_text"].fillna("")
        print(
            f"Warning: {missing} submission ids missing from test.csv; filled with empty text."
        )

    test_texts = df_submit_base["comment_text"].values

    if training_ok:
        all_test_preds = []
        for col in tqdm(y_cols, desc="Test predict columns"):
            model = model_dict[col]
            toxic_preds = fasttext_predict_proba_batched(
                model, test_texts, batch_size=100000, pos_label="__class__1"
            )
            all_test_preds.append(toxic_preds)

        all_test_preds_array = np.transpose(np.array(all_test_preds))
        df_submit_base[y_cols] = all_test_preds_array
        submission = df_submit_base[["id"] + y_cols].copy()
    else:
        submission = sample_submission[["id"] + y_cols].copy()

    if list(submission.columns) != ["id"] + y_cols:
        raise ValueError("Submission columns are not in the required order.")
    if submission.shape[0] != sample_submission.shape[0]:
        raise ValueError(
            f"Submission row count {submission.shape[0]} != sample_submission row count {sample_submission.shape[0]}"
        )
    if not submission["id"].equals(sample_submission["id"]):
        raise ValueError(
            "Submission id order does not match sample_submission id order."
        )

except Exception as e:
    print(
        "Warning: submission generation failed; writing sample_submission fallback. Error:",
        repr(e),
    )
    submission = sample_submission[["id"] + y_cols].copy()

submission.to_csv(sub_path, index=False)

if (not os.path.exists(sub_path)) or (os.path.getsize(sub_path) < 1000):
    raise RuntimeError("Submission file was not written correctly.")

print(
    "Wrote:", sub_path, "shape:", submission.shape, "bytes:", os.path.getsize(sub_path)
)
submission.head()

Test predict columns:   0%|          | 0/6 [00:00<?, ?it/s]

Batched predict:   0%|          | 0/2 [00:00<?, ?it/s]

Batched predict:   0%|          | 0/2 [00:00<?, ?it/s]

Batched predict:   0%|          | 0/2 [00:00<?, ?it/s]

Batched predict:   0%|          | 0/2 [00:00<?, ?it/s]

Batched predict:   0%|          | 0/2 [00:00<?, ?it/s]

Batched predict:   0%|          | 0/2 [00:00<?, ?it/s]

Wrote: /kaggle/working/submission.csv shape: (153164, 7) bytes: 13846438


,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,00001cee341fdb12,0.948165,0.040856,0.835494,0.001711,0.724880,0.034110
1,0000247867823ef7,0.000513,0.000010,0.000010,0.000010,0.000010,0.000010
2,00013b17ad220c46,0.007826,0.008857,0.007587,0.008072,0.005070,0.012442
3,00017563c3f7919a,0.004478,0.000598,0.003183,0.000010,0.002900,0.000513
4,00017695ad8997eb,0.017996,0.000390,0.003493,0.000010,0.001465,0.000454
